In [ ]:
# ============================================================================
# SAVE DIAGRAM AS HTML (INTERACTIVE)
# ============================================================================

# Create export directory
export_dir = Path("./sankey_html_exports")
export_dir.mkdir(exist_ok=True)

# Generate filename from impact category and system
safe_impact = impact_category.replace("/", "_").replace(" ", "_")
html_filename = f"{safe_impact}_sankey_visualization.html"
html_path = export_dir / html_filename

# Save as interactive HTML
fig.write_html(str(html_path))

print(f"✓ Saved interactive HTML: {html_path}")
print(f"  Open this file in your browser to explore the diagram interactively")

# ============================================================================
# OPTIONAL: SAVE AS STATIC IMAGE (requires kaleido)
# ============================================================================

# Uncomment these lines if you have kaleido installed:
# try:
#     png_filename = f"{safe_impact}_sankey_visualization.png"
#     png_path = export_dir / png_filename
#     fig.write_image(str(png_path))
#     print(f"✓ Also saved as PNG: {png_path}")
# except Exception as e:
#     print(f"Note: PNG export requires 'pip install kaleido' ({e})")

print(f"\n📁 Export directory: {export_dir.resolve()}")

## Bonus: Save and Export the Diagram

You can save the diagram as an interactive HTML file or static image.

## How to Interpret the Sankey Diagram

### Visual Elements

1. **Nodes (Rectangles)**
   - Represent individual processes in your supply chain
   - **Color intensity** indicates the magnitude of impact (darker/redder = higher impact)
   - **Height** is proportional to the contribution to the selected impact category
   - Hover over nodes to see exact values

2. **Edges (Flow Lines)**
   - Show connections between processes (material/energy flows or impact contributions)
   - **Width** represents the percentage share of the upstream impact flowing from supplier to receiver
   - Hover over edges to see the exact percentage and process names

3. **Flow Direction**
   - Flows move from left to right
   - Left side: upstream suppliers (processes that provide materials)
   - Right side: downstream receivers (processes that use materials)

### Interpretation Tips

- **Identify hotspots**: Thicker flows = larger impact contributions
- **Trace supply chains**: Follow flows to understand material/energy paths
- **Compare impacts**: Compare node colors to see which processes have highest impact
- **Interactive exploration**: Hover and click to drill into specific flows

### Configuration Parameters Used

The visualization is based on these settings from `result_calculation_explained.py`:

```
SANKEY_MODE = {mode}              # {mode_description}
SANKEY_TOP_FLOWS = {max_nodes_configured}        # Max processes to show
SANKEY_MAX_DEPTH = 3              # Upstream levels (not currently supported)
```

In [ ]:
# ============================================================================
# DISPLAY SANKEY DIAGRAM
# ============================================================================

print("🎨 SANKEY DIAGRAM VISUALIZATION")
print("=" * 70)
print("\n📊 Rendering interactive diagram...")

# Display the figure
fig.show()

print("\n✓ Diagram displayed!")

## Section 7: Display and Interpret the Diagram

Display the final interactive Sankey diagram and explain how to interpret the flows and relationships.

In [ ]:
# ============================================================================
# CUSTOMIZE LAYOUT AND APPEARANCE
# ============================================================================

# Determine mode description
mode_description = {
    1: "Flow-based (Environmental flows)",
    2: "Impact-based (Impact contributions)"
}.get(mode, f"Mode {mode}")

# Update layout with styling
fig.update_layout(
    title=dict(
        text=f"<b>LCA Sankey Diagram: {impact_category}</b><br><sub>{mode_description}</sub>",
        x=0.5,
        xanchor="center",
        font=dict(size=16, color="#2c3e50")
    ),
    font=dict(
        size=12,
        family="Arial, sans-serif",
        color="#2c3e50"
    ),
    height=700,
    width=1400,
    plot_bgcolor="white",
    paper_bgcolor="#f8f9fa",
    margin=dict(l=20, r=20, t=100, b=20),
    
    # Add annotations for context
    annotations=[
        dict(
            text=f"<i>Configuration: SANKEY_MODE={mode} | MAX_NODES={max_nodes_configured}</i><br>"
                 f"<i>Nodes: {len(nodes)} processes | Edges: {len(edges)} flows</i>",
            xref="paper", yref="paper",
            x=0.5, y=-0.05,
            showarrow=False,
            font=dict(size=10, color="gray")
        )
    ]
)

print("✓ Layout customized:")
print(f"  • Title: {impact_category} ({mode_description})")
print(f"  • Dimensions: 1400x700 pixels")
print(f"  • Font: Arial, 12pt")
print(f"  • Configuration info displayed in annotations")

## Section 6: Customize Sankey Diagram Appearance

Apply customizations such as colors, fonts, titles, and layout adjustments to enhance visualization.

In [ ]:
# ============================================================================
# CREATE PLOTLY SANKEY FIGURE
# ============================================================================

# Create the Sankey diagram using Plotly's go.Sankey object
fig = go.Figure(data=[go.Sankey(
    # NODE CONFIGURATION
    node=dict(
        pad=15,                          # Space between nodes
        thickness=20,                    # Node thickness
        line=dict(color="black", width=0.5),
        label=node_labels,               # Node names (process providers)
        color=node_colors,               # Node colors (based on impact intensity)
        customdata=node_total_results,   # Data for hover
        hovertemplate='<b>%{label}</b><br>Total Impact: %{customdata:.4f}<extra></extra>'
    ),
    # EDGE CONFIGURATION
    link=dict(
        source=edge_sources,             # Source node indices
        target=edge_targets,             # Target node indices
        value=edge_values,               # Flow values (% share)
        color="rgba(200, 200, 200, 0.4)",
        hovertemplate='%{source.label} → %{target.label}<br>Share: %{value:.2f}%<extra></extra>'
    )
)])

print("✓ Sankey diagram created with:")
print(f"  • {len(node_labels)} nodes (processes)")
print(f"  • {len(edge_sources)} edges (flows)")
print(f"  • Interactive hover information enabled")

## Section 5: Create Sankey Diagram Visualization

Use Plotly's Sankey graph object to create the diagram with source indices, target indices, values, and node labels.

In [ ]:
# ============================================================================
# PREPARE NODE DATA FOR PLOTLY
# ============================================================================

# Node colors based on total_result magnitude (impact intensity)
max_total = max(node_total_results) if node_total_results else 1
normalized_values = [v / max_total for v in node_total_results]

# Create RGB colors: intensity from blue (low) to red (high)
node_colors = []
for norm_val in normalized_values:
    # Red intensity increases with impact
    r = int(255 * norm_val)
    g = int(150 * (1 - norm_val))
    b = int(100 * (1 - norm_val))
    node_colors.append(f"rgb({r},{g},{b})")

# Create hover text with detailed information
node_hover_text = []
for i, node in enumerate(nodes):
    text = f"<b>{node['provider']}</b><br>"
    text += f"Direct Impact: {node['direct_result']:.4f}<br>"
    text += f"Total Impact: {node['total_result']:.4f}"
    node_hover_text.append(text)

print("✓ Node data prepared:")
print(f"  • Colors generated based on impact magnitude")
print(f"  • Hover text with direct and total impact values")
print(f"  • {len(node_colors)} colors for {len(node_labels)} nodes")

# ============================================================================
# PREPARE EDGE DATA FOR PLOTLY
# ============================================================================

# Create hover text for edges (showing flow information)
edge_hover_text = []
for i, edge in enumerate(edges):
    source_label = node_labels[edge_sources[i]] if edge_sources[i] < len(node_labels) else "Unknown"
    target_label = node_labels[edge_targets[i]] if edge_targets[i] < len(node_labels) else "Unknown"
    text = f"{source_label} → {target_label}<br>"
    text += f"Share: {edge_values[i]:.2f}%"
    edge_hover_text.append(text)

print("\n✓ Edge data prepared:")
print(f"  • Source nodes: {len(edge_sources)}")
print(f"  • Target nodes: {len(edge_targets)}")
print(f"  • Values (% share): {len(edge_values)} edges")
print(f"  • Hover text with flow descriptions")

## Section 4: Prepare Data for Sankey Diagram

Transform extracted parameters into Plotly Sankey format: source indices, target indices, values, and node labels with colors.

In [ ]:
# ============================================================================
# EXTRACT PARAMETERS FOR PLOTLY SANKEY
# ============================================================================
# These parameters were defined in result_calculation_explained.py:
#   - SANKEY_MODE: 1 (Flow-based) or 2 (Impact-based)
#   - SANKEY_TOP_FLOWS: Maximum nodes for flow-based (default: 10)
#   - SANKEY_TOP_IMPACTS: Maximum nodes for impact-based (default: 5)

# Extract from Sankey data
nodes = sankey_data.get("nodes", [])
edges = sankey_data.get("edges", [])
mode = sankey_data.get("mode", 1)
impact_category = sankey_data.get("impact_category", "Unknown")
max_nodes_configured = sankey_data.get("max_nodes", len(nodes))

# ============================================================================
# PARAMETERS FOR SANKEY CONSTRUCTION
# ============================================================================

# NODE PARAMETERS
node_labels = [n["provider"] for n in nodes]
node_direct_results = [n["direct_result"] for n in nodes]
node_total_results = [n["total_result"] for n in nodes]

# EDGE PARAMETERS (flow connections)
edge_sources = [e["node_index"] for e in edges]           # Receiving process
edge_targets = [e["provider_index"] for e in edges]       # Supplying process
edge_values = [e["upstream_share"] * 100 for e in edges]  # Convert to percentage

print("=" * 70)
print("EXTRACTED PARAMETERS")
print("=" * 70)
print(f"\n🎯 Sankey Mode: {mode} ({'Flow-based' if mode == 1 else 'Impact-based'})")
print(f"📌 Impact Category: {impact_category}")
print(f"⚙️  Maximum Nodes Configured: {max_nodes_configured}")
print(f"\n📊 NODE PARAMETERS ({len(nodes)} nodes):")
print(f"   • Labels: {len(node_labels)} process names")
print(f"   • Direct Results: min={min(node_direct_results):.4f}, max={max(node_direct_results):.4f}")
print(f"   • Total Results: min={min(node_total_results):.4f}, max={max(node_total_results):.4f}")
print(f"\n🔗 EDGE PARAMETERS ({len(edges)} edges):")
print(f"   • Sources: {edge_sources[:3]}... (connecting from nodes)")
print(f"   • Targets: {edge_targets[:3]}... (connecting to nodes)")
print(f"   • Values (% share): min={min(edge_values):.2f}%, max={max(edge_values):.2f}%")

## Section 3: Extract Parameters from result_calculation_explained

These parameters define the Sankey diagram structure based on the result calculation configuration.

In [ ]:
# Load Sankey JSON data
with open(selected_file, "r", encoding="utf-8") as f:
    sankey_data = json.load(f)

# ============================================================================
# EXPLORE DATA STRUCTURE
# ============================================================================

print("=" * 70)
print("SANKEY DATA STRUCTURE")
print("=" * 70)

# Display top-level keys
print(f"\n📋 Available Keys: {list(sankey_data.keys())}")

# Display metadata
print(f"\n📊 Impact Category: {sankey_data.get('impact_category')}")
print(f"📊 Mode: {sankey_data.get('mode')} ", end="")
mode_desc = "Flow-based" if sankey_data.get('mode') == 1 else "Impact-based"
print(f"({mode_desc})")
print(f"📊 Max Nodes: {sankey_data.get('max_nodes')}")

# Display nodes information
nodes = sankey_data.get("nodes", [])
print(f"\n🔵 Nodes ({len(nodes)} total):")
print("   index | provider                          | direct_result  | total_result")
print("   " + "-" * 68)
for node in nodes[:5]:  # Show first 5
    idx = node.get("index", "?")
    provider = node.get("provider", "Unknown")[:30].ljust(30)
    direct = f"{node.get('direct_result', 0):.4f}"
    total = f"{node.get('total_result', 0):.4f}"
    print(f"   {idx:<5} | {provider} | {direct:>14} | {total:>12}")
if len(nodes) > 5:
    print(f"   ... and {len(nodes) - 5} more nodes")

# Display edges information
edges = sankey_data.get("edges", [])
print(f"\n🔗 Edges ({len(edges)} total):")
print("   source_idx → target_idx | upstream_share")
print("   " + "-" * 40)
for edge in edges[:5]:  # Show first 5
    src = edge.get("node_index", "?")
    tgt = edge.get("provider_index", "?")
    share = f"{edge.get('upstream_share', 0)*100:.2f}%"
    print(f"   {src:>2} → {tgt:<2} | {share:>14}")
if len(edges) > 5:
    print(f"   ... and {len(edges) - 5} more edges")

In [ ]:
# ============================================================================
# CONFIGURATION - PATHS
# ============================================================================

# Current working directory (LCI/RESULTS/)
results_dir = Path.cwd()

# List all available Sankey JSON files
sankey_files = list(results_dir.glob("*_sankey.json"))

print(f"📁 Results Directory: {results_dir}")
print(f"🔍 Found {len(sankey_files)} Sankey file(s):")
for file in sorted(sankey_files):
    print(f"   • {file.name}")

# Select the first file for this example (or specify manually)
if sankey_files:
    selected_file = sorted(sankey_files)[0]
    print(f"\n✓ Selected for visualization: {selected_file.name}")
else:
    print("⚠ No Sankey files found! Run result_calculation_explained.py first.")

## Section 2: Load and Explore Sankey Data

The Sankey JSON files are generated by `result_calculation_explained.py` and contain the data structure needed for visualization. These files follow the naming pattern: `{system}_{method}_sankey.json`

In [ ]:
import json
import plotly.graph_objects as go
import pandas as pd
from pathlib import Path
import os

# Display configuration
from IPython.display import display, HTML

print("✓ Libraries imported successfully")

## Section 1: Import Required Libraries

Import necessary libraries for data manipulation, visualization, and file handling.

# Sankey Diagram Visualization from LCA Results

This notebook demonstrates how to visualize Sankey diagrams using the parameters and data structures generated by `result_calculation_explained.py`.

**Key Concepts:**
- **Nodes**: Individual processes in the supply chain (suppliers, components, etc.)
- **Edges**: Connections showing material/energy flows or impact contributions between processes
- **Modes**: 
  - Mode 1: Flow-based (shows environmental flows between processes)
  - Mode 2: Impact-based (shows impact contributions between processes)